In [ ]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Conv2D,Dropout,Flatten,Activation,BatchNormalization,AveragePooling2D,MaxPooling2D
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from tensorflow.python.keras import *
from tensorflow.python.keras.layers import *
import tensorflow as tf
import numpy as np
import random
from tensorflow.keras import backend as K


M = 128                     # 每个数据样本的比特长度
batch_size = 512           # 批次大小
block_length = M           # 每个块的长度，也设为128
len_h = 8                  # 信道的脉冲响应长度（代表信道记忆长度）--------修改这个
EbNo_train = 20            # 训练时的信噪比（以线性比例表示，不是 dB）
R = 0.5                    # 编码率 R（码率）
std = (np.sqrt(1/(2*R*EbNo_train)))   # 高斯噪声标准差
len_info = 40              # 有效信息比特长度（可能用于提取一部分数据）

N_train = 160000
data = np.random.binomial(1, 0.5, [N_train, M, 1])
N_val = 1500
val_data = np.random.binomial(1, 0.5, [N_val, M, 1])
N_test = 5000
test_data = np.random.binomial(1, 0.5, [N_test, M, 1])

In [ ]:
#y = x * h[0]；后续每个路径都延迟i个时间步再加权叠加。
def convolution(x, h):
  y = x * tf.cast(tf.reshape(h[:, 0, 0], [-1, 1, 1]),tf.float32)
  for i in range(1, len_h):
      cur = x * tf.cast(tf.reshape(h[:, i, 0], [-1, 1, 1]),tf.float32)
      cur = tf.concat([cur[:, -i:, :], cur[:, :-i, :]], 1)
      y += cur
  return y


#生成复值多径信道冲激响应的实部或虚部（二维向量），按功率分布 PDP 加权。
def sample_h_mp(sample_size, PDP):
    """ Generate real and imagary part of channel """
    h = 1 / np.sqrt(2) * np.random.normal(size=sample_size)
    for i in range(len(h)):
        h[i] = h[i] * np.sqrt(PDP).reshape([-1, 1])
    return h
'''

def sample_h_mp(sample_size, PDP):
    batch_size, L, two = sample_size
    assert two == 2, "最后一维必须为 2（代表实部和虚部）"
    PDP = np.array(PDP, dtype=float)
    assert PDP.ndim == 1 and PDP.size == L, "PDP 长度必须等于 len_h"

    # 1) 先生成 (batch_size, L, 2) 的高斯随机数 (实部+虚部)，各分量 ~ N(0, 1/2)
    #    这样复信道 h 的每个分量幅度平方期望为 1
    h = np.random.randn(batch_size, L, 2) / np.sqrt(2)

    # 2) 按照 PDP 对每条路径乘上 sqrt(PDP[i])，保证功率分布正确
    #    把 PDP 扩展为 (1, L, 1)，让它能在批次和实/虚维度上广播
    sqrtPDP = np.sqrt(PDP).reshape(1, L, 1)
    h *= sqrtPDP

    return h
'''
#生成单位均匀的功率延迟分布（PDP），等价于所有路径均匀分布。
def bilinear_mul(x, info):
    x_reshape = tf.reshape(x, [-1, (block_length+len_h)*2, 1])
    info_reshape = tf.reshape(info, [-1, 1, len_info])
    mat_reshape = tf.matmul(x_reshape, info_reshape)
    return tf.reshape(mat_reshape, [-1, (block_length+len_h), len_info*2])

#从 data 数组中顺序生成训练批数据，若数据用尽则重新采样
def generate_batch_data(batch_size):
    global start_idx, data
    if start_idx + batch_size >= N_train:
        start_idx = 0
        data = np.random.binomial(1, 0.5, [N_train, block_length, 1])
    batch_x = data[start_idx:start_idx + batch_size]
    start_idx += batch_size
    # print("start_idx", start_idx)
    return batch_x
'''
def generate_PDP(L):
    """ Generate the PDP for channel generation """
    PDP = np.ones(L)
    PDP = PDP / sum(PDP)
    print("PDP:", PDP)
    return PDP
'''
def generate_PDP(L):
    # 先对后面 L-1 个元素随机采样
    rest = np.random.rand(L - 1)
    max_rest = rest.max()
    first = max_rest + np.random.rand()
    arr = np.concatenate(([first], rest))
    PDP = arr / arr.sum()
    print("PDP:", PDP)
    return PDP

#h = sample_h_mp(batch_size)



def end_to_end():
  input_shape = (M, 1)
  input_h_shape = (len_h,2)
  input_sdv_shape = (1,)
  input = Input(shape = input_shape)
  input_h = Input(shape = input_h_shape)
  input_sdv = Input(shape = input_sdv_shape)
  ############## encoder
  x = Conv1D(256, 5, padding = 'same', activation = 'relu')(input)
  x = Conv1D(128, 3, padding = 'same', activation = 'relu')(x)
  x = Conv1D(64, 3, padding = 'same', activation = 'relu')(x)
  x = Conv1D(2, 3, padding = 'same', activation = 'relu')(x)
  layer_4_normalized = tf.math.scalar_mul(tf.math.sqrt(tf.cast(batch_size*block_length, tf.float32)),
                                           tf.math.l2_normalize(x))


  ##############multi_layer
  x_pad = tf.pad(layer_4_normalized, tf.constant([[0, 0], [0, len_h], [0, 0]]))

  #h_r = tf.reshape(h[:, :, 0], [-1, len_h, 1])
  #h_i = tf.reshape(h[:, :, 1], [-1, len_h, 1])
  h_r = tf.reshape(input_h[:, :, 0], [-1, len_h, 1])
  h_i = tf.reshape(input_h[:, :, 1], [-1, len_h, 1])
  x_r = tf.reshape(x_pad[:, :, 0], [-1, block_length + len_h, 1])
  x_i = tf.reshape(x_pad[:, :, 1], [-1, block_length + len_h, 1])
  o_r = convolution(x_r, h_r) - convolution(x_i, h_i)
  o_i = convolution(x_r, h_i) + convolution(x_i, h_r)
  output = tf.concat([o_r, o_i], -1)
  #output = tf.concat([o_r, o_i], -1)[:,int(len_h/2):int(len_h/2)+block_length,:]
  noise = tf.random.normal(shape=tf.shape(output), mean=0.0, stddev=input_sdv, dtype=tf.float32)
  output_noisy = output+noise

  ###############channel estimation
  x = Conv1D(256, 5, padding = 'same', activation = 'relu')(output_noisy)
  x = Conv1D(128, 3, padding = 'same', activation = 'relu')(x)
  x = Conv1D(64, 3, padding = 'same', activation = 'relu')(x)
  x = Conv1D(32, 3, padding = 'same', activation = 'relu')(x)
  x = Conv1D(3, 3, padding = 'same', activation = 'relu')(x)
  conv4_flat = tf.reshape(x, [-1, 3 * (block_length + len_h)])
  x = Dense(100, activation = 'linear')(conv4_flat)
  estimated_channel = Dense(40, activation = 'linear')(x)


  ##############Decoder
  x_combine = bilinear_mul(output_noisy, estimated_channel)
  x = Conv1D(256, 5, padding = 'same', activation = 'relu')(x_combine)
  #x = Conv1D(256, 5, padding = 'same', activation = 'relu')(output_noisy)
  x_ori_1 = Conv1D(128, 5, padding = 'same', activation = 'relu')(x)
  x = Conv1D(128, 5, padding = 'same', activation = 'relu')(x_ori_1)
  x = Conv1D(128, 5, padding = 'same')(x)
  x = x + x_ori_1
  x = tf.nn.relu(x)
  x_ori_2 = Conv1D(64, 5, padding = 'same', activation = 'relu')(x)
  x = Conv1D(64, 5, padding = 'same', activation = 'relu')(x_ori_2)
  x = Conv1D(64, 3, padding = 'same')(x)
  x = x + x_ori_2
  x = tf.nn.relu(x)
  x = Conv1D(32, 3, padding = 'same', activation = 'relu')(x)
  pred_logits = Conv1D(1, 3, padding = 'same')(x)
  pred_prob = tf.nn.sigmoid(pred_logits)
  pred_logits_reshape = pred_logits[:, 0:block_length, :]
  pred_prob_reshape = pred_prob[:, 0:block_length, :]

  model  = Model(inputs = [input,input_h,input_sdv], outputs = [pred_logits_reshape,pred_prob_reshape])
  return model

def to_tensor(x):
    return tf.convert_to_tensor(x, dtype=tf.float32)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate = 0.0001)
mse_loss_fn = tf.keras.losses.MeanSquaredError() #损失函数1
binary_crossentropy_fn = tf.keras.losses.BinaryCrossentropy() #损失函数2
model = end_to_end()


#超参数
number_steps = int(8e3) + 1       # 训练步数：共8000次
testing_step = 1000              # 每1000步测试一次
PDP_test0 = [0.29240794, 0.11956707, 0.10309964, 0.10567278, 0.09299257, 0.10077287, 0.0103536, 0.17513352] # 生成信道功率分布（PDP）
PDP_train1 = generate_PDP(len_h)

EbNo_train = 20
EbNo_train = 10 ** (EbNo_train / 10)  # 线性SNR
start_idx = 0

#每 100 步打印一次损失，用于监控训练过程(我需要修改)
for step in range(number_steps):
  if step % 100 == 0 and step>0:
    print("Training step:", step)
    print(loss)


#获取训练数据与信道
  batch_x = generate_batch_data(batch_size)
  h = sample_h_mp((batch_size, len_h, 2), PDP_test0)
  #print(h)

  #if step == 0:
    #model = end_to_end()
  #elif step % 100 ==0 and step != 0:
    #model = end_to_end()
    #model.set_weights(weights_previous)


#正向传播与损失计算
  with tf.GradientTape() as tape:
    y_pred_logits, y_pred = model([batch_x,h,np.sqrt(1/(2*R*EbNo_train))])
    batch_x_tensor = tf.cast(to_tensor(batch_x),tf.float32)
    loss = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(logits=y_pred_logits, labels=batch_x_tensor))

#反向传播与参数更新
  grads_api = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads_api,model.trainable_variables))
  #optimizer.minimize(loss,model.trainable_variables)
  #if step % 100 == 99:
    #weights_previous = model.get_weights()

 #test
  if step%testing_step == 0 and step > 0:
    h_test = sample_h_mp((batch_size, len_h, 2),PDP_train1)
    # 和h取一样的PDP，得到match性能。
    #记录BER
    EbNodB_range = np.arange(0, 25, 5)
    #EbNodB_range = np.arange(10, 21)
    ber = np.ones(len(EbNodB_range))
    wer = np.ones(len(EbNodB_range))
    for n in range(0, len(EbNodB_range)):
        EbNo = 10.0 ** (EbNodB_range[n] / 10.0)
        ber_list = list()
        wer_list = list()
        idx = 0
        while idx+batch_size < N_test:
          test_data_batch = test_data[idx:idx+batch_size]
          y_pred_logits_test, y_pred_test = model([test_data_batch,h_test,np.sqrt(1/(2*R*EbNo))])
          test_data_batch_tensor = to_tensor(test_data_batch)
          accuracy = 1 - tf.reduce_mean(tf.cast(tf.abs(y_pred_test - test_data_batch_tensor) < 0.5, tf.float32))
          ber_list.append(1 - accuracy)
          idx+=batch_size
        ber[n]=1-np.mean(np.asarray(ber_list))
        print("111111111111111111111111111111111111111")
        print('SNR:', EbNodB_range[n], 'Len:', block_length, 'BER:', ber[n])
  #if step%10000 == 0 and step >0:
    #model.save_weights('/content/gdrive/MyDrive/wireless communication/blind_convolition/8_path')

PDP: [0.29240794 0.11956707 0.10309964 0.10567278 0.09299257 0.10077287
 0.0103536  0.17513352]
Training step: 100
tf.Tensor(0.5532192, shape=(), dtype=float32)
Training step: 200
tf.Tensor(0.51664287, shape=(), dtype=float32)
Training step: 300
tf.Tensor(0.5057775, shape=(), dtype=float32)
Training step: 400
tf.Tensor(0.48974437, shape=(), dtype=float32)
Training step: 500
tf.Tensor(0.4850092, shape=(), dtype=float32)
Training step: 600
tf.Tensor(0.4765662, shape=(), dtype=float32)
Training step: 700
tf.Tensor(0.47043025, shape=(), dtype=float32)
Training step: 800
tf.Tensor(0.46057215, shape=(), dtype=float32)
Training step: 900
tf.Tensor(0.4465196, shape=(), dtype=float32)
Training step: 1000
tf.Tensor(0.4376649, shape=(), dtype=float32)
111111111111111111111111111111111111111
SNR: 0 Len: 128 BER: 0.4481337070465088
111111111111111111111111111111111111111
SNR: 5 Len: 128 BER: 0.383880615234375
111111111111111111111111111111111111111
SNR: 10 Len: 128 BER: 0.3044365644454956
111111111

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-5-63a258224a9c>", line 41, in <cell line: 0>
    grads_api = tape.gradient(loss, model.trainable_variables)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/eager/backprop.py", line 1066, in gradient
    flat_grad = imperative_grad.imperative_grad(
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/eager/imperative_grad.py", line 67, in imperative_grad
    return pywrap_tfe.TFE_Py_TapeGradient(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/eager/backprop.py", line 148, in _gradient_function
    return grad_fn(mock_op, *out_grads)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

TypeError: object of type 'NoneType' has no len()